# Model Evaluation

This tutorial walks through basic evaluations on a MR-LFADS model trained on the memory network synthetic dataset, including assessing model fit quality, inferred connectivity and communication content.

## 1. Load Model

Once MR-LFADS is trained, load the model by specifying the absolute path to the `main.yaml` configuration file:

In [ ]:
import os
import config.paths as path

path_to_config = NotImplemented # e.g. os.path.join(path.resultpath, '01_memory_network', 'configs', 'main.yaml')

In [ ]:
from mrlfads.run import load

state_dict = load(
    config_path = path_to_config,
    validate = True,
)
model = state_dict['model']
dm = state_dict['datamodule']
metrics = state_dict['metrics']

## 2. Accessing Quantities of Interest

Below is a set of quantities relevant to inference:

In [ ]:
# Data passed into model
session = 0 # session of interest
data = model.current_batch[session]       # stores raw spike counts and external inputs
metadata = model.current_info[session]    # stores other metadata passed in through the datamodule

# Reconstruction quantities
area_name = NotImplemented # e.g. 'A0'
output_params = model.outputs[area_name][session]      # reconstructed output parameters (e.g. rates for Poisson, mean + std for Gaussian)
rates = output_params.exp()                            # For Poisson, the rates need to be exponentiated

# Inferred quantities
ic_params = model.save_var[area_name].ic_params        # initial condition parameters (mean, std)
hps = model.hparams # model hyperparameters
ahps = model.areas[area_name].hparams # single area specific hyperparameters
inputs = model.save_var[area_name].inputs.cpu().detach()
_, inferred_inputs, comm = torch.split(inputs, [ahps.ci_enc_dim, ahps.co_dim, ahps.com_dim * hps.num_other_areas], dim=2)    # inferred input and communciation

## 3. Reconstruction Quality

Inference should only be done on models with good reconstruction fit. It is advised to perform a hyperparameter search and use the model that achieves the lowest hold-out neuron loss (see `tutorials/03_Hyperparameter_Search.ipynb`). 

Tools for visualization of the model's predicted rates for held-in and held-out neurons are provided in `mrlfads/evals/visualization.py`:

In [ ]:
import numpy 

# Adjust number of neurons to plot
neurons_to_plot = 4

# Optionally chooses the neurons with the highest firing rate, as
# models typically predict high firing rate neurons more accurately
indices = {}
for area_name in model.area_names:
    data = model.current_batch[0].encod_data[area_name].cpu().detach().numpy()
    top_neurons = np.flip(np.argsort(np.linalg.norm(data, axis=(0, 1))))
    indices[area_name] = top_neurons[:neurons_to_plot]

In [ ]:
%matplotlib inline
from mrlfads.evals.visualization import plot_reconstruction

# Plot held-in neurons
plot_reconstruction(
    model,
    indices=indices,
    smooth=False,      # for spiking data, it is helpful to smooth the spikes for comparison
)
plt.show()

In [ ]:
# Plot held-out neurons
plot_holdout_reconstruction(
    model,
    smooth=False,
)

The following primary metrics are useful for assessing reconstruction performance:

* Reconstruction loss for held-in neurons
* Reconstruction loss for held-out neurons
* McFadden $r^2$ for held-in neurons
* McFadden $r^2$ for held-out neurons

For McFadden $r^2$, values in the range of $0.3$–$0.4$ are generally considered indicative of a good fit.

Other secondary metrics that may be helpful for eliminating overfitted models include:

* KL(u) or KL(m) loss: if the numbers are orders of magnitude larger than reconstruction loss, the model may have reached a local minimum.

Note that all quantities are computed over the validation dataset.

In [ ]:
from mrlfads.evals.metrics import fit_metrics

summary = fit_metrics(model, metrics)
for key, val in summary.items():
    print(key, ':', val)

## 4. Connectivity Diagram

A quick overview of the inferred connectivity structure can be visualized with `plot_anatomy`, which plots connections based on the communication norm values:

In [ ]:
from mrlfads.evals.visualization import plot_anatomy

plot_anatomy(model)

A more detailed view of the communication, represented as a ragged list of shape `(target area, source area, batch size, time size, channel size)`, can be visualized using `volume`:

In [ ]:
from mrlfads.evals.metrics import volume

comm = volume(model)[0]

## 5. Communication Content

To assess whether communication from a source area to a target area encodes a task-relevant variable, a simple first-pass analysis is to fit a linear regression model. A rough outline is as follows:

In [ ]:
# If the task variable is included in the datamodule as metadata,
# it is easier to align trials with the corresponding model outputs.
task_var = NotImplemented

# (source, target) communication of interest
source_area = NotImplemented
target_area = NotImplemented

In [ ]:
from mrlfads.evals.metrics import PolyRegression

idx_src = model.area_names.index(source_area)
idx_tar = model.area_names.index(target_area)
X = comm[idx_tar][idx_src]

# Fit (X, y), where X, y is a 3-dimensional numpy array of shape (batch size, time size, fea size)
reg = PolyRegression(1)
reg.ffit(X, task_var)
score = reg.fscore(X, task_var)
print('Score of predicting task variable from communication:', score)

For the memory network, the task variable (i.e. stimulus to each area) can be found in the meta-data:

In [ ]:
# Allow datamodule to load the meta-data
dm.hparams.ignore_info = False
dm.setup()

# Input to area A0, A2, A2
task_vars = data[0][1]["inp"].cpu().detach()
task_var_0, task_var_1, task_var_2 = torch.split(task_vars, [2, 3, 4], dim=2)